# OneDoor Support-Need & Urgency Classifier

**Research Question:** *Which urgent cases does the model miss, and for what kinds of messages?*

This notebook trains and evaluates a multi-label text classifier on a **100% synthetic Addis Ababa dataset** (`synthetic_dataset.json`) covering six support categories:
1. `medical_support`
2. `psychological_support`
3. `protection_safeguarding`
4. `legal_support`
5. `sexual_violence_support`
6. `social_support`

Plus a separate binary **urgency indicator** (non-diagnostic triage aid). All results below are evaluated on a strictly held-out 30% test split (60 examples) never seen during vocabulary building or training.

In [1]:
import json
from pathlib import Path
from collections import Counter

with open('synthetic_dataset.json', 'r', encoding='utf-8') as f:
    records = json.load(f)

from train import stratified_split, length_bucket, LABELS
for r in records:
    r['message_length'] = length_bucket(r['text'])

train_rows, test_rows = stratified_split(records, test_ratio=0.30, seed=42)
print(f'Loaded {len(records)} synthetic Addis Ababa examples.')
print(f'Train split: {len(train_rows)} | Held-out Test split: {len(test_rows)}')
print(f'Test urgent cases: {sum(r["is_urgent"] for r in test_rows)} / {len(test_rows)}')

Loaded 200 synthetic Addis Ababa examples.
Train split: 140 | Held-out Test split: 60
Test urgent cases: 37 / 60


## Training Pipeline (Scikit-Learn & Portable JSON Export)

We vectorize descriptions using **TF-IDF (unigrams + bigrams, sublinear TF, `min_df=2`)** and train **One-vs-Rest L2-regularized Logistic Regression** classifiers. We export the trained model with `joblib` (when scikit-learn is installed) and export vocabulary, IDF weights, coefficients, and intercepts to `model_weights.json` for deterministic TypeScript inference in the web backend.

In [2]:
# Standard scikit-learn reference pipeline (executed via train.py):
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.multiclass import OneVsRestClassifier
# import joblib
import train
train.main()

Training & held-out evaluation complete.
Train size: 140, Held-out test size: 60, Vocab size: 739
Majority Macro-F1: 0.1159 | Urgency F1: 0.7629
Keyword  Macro-F1: 0.6757 | Urgency F1: 0.6452
TF-IDF   Macro-F1: 0.6622 | Urgency F1: 0.7955
TF-IDF Urgency FN=2, FP=16, Recall=0.9459


## Held-Out Baseline Comparison & Error Slicing

Below we inspect the exact held-out test metrics from `evaluation_results.json`, including False Negatives, False Positives, slices by **language style** (`direct`, `euphemistic`, `distressed_fragmented`, `third_person`) and **message length** (`short`, `medium`, `long`).

In [3]:
with open('evaluation_results.json', 'r', encoding='utf-8') as f:
    ev = json.load(f)

print('=== Urgency Detection (Held-Out Test Set, n=60) ===')
for name, label in [('majority_class', 'Majority Class '), ('keyword_rule', 'Keyword Rule   '), ('trained_tfidf_logreg', 'TF-IDF LogReg  ')]:
    u = ev['baseline_comparison'][name]['urgency']
    print(f"{label}: Recall={u['recall']:.4f} | Precision={u['precision']:.4f} | F1={u['f1']:.4f} | FN={u['fn']:<2} | FP={u['fp']}")

print('\n=== Performance by Language Style ===')
for style, s in ev['performance_by_language_style'].items():
    print(f"{style:<22} (n={s['count']}): Macro-F1={s['multilabel_macro_f1']:.4f} | Urgency Recall={s['urgency_recall']:.4f} | Urgency FN={s['urgency_false_negatives']}")

print('\n=== Performance by Message Length ===')
for length, s in ev['performance_by_message_length'].items():
    print(f"{length:<6} (n={s['count']}): Macro-F1={s['multilabel_macro_f1']:.4f} | Urgency Recall={s['urgency_recall']:.4f} | Urgency FN={s['urgency_false_negatives']}")

=== Urgency Detection (Held-Out Test Set, n=60) ===
Majority Class : Recall=1.0000 | Precision=0.6167 | F1=0.7629 | FN=0  | FP=23
Keyword Rule   : Recall=0.5405 | Precision=0.8000 | F1=0.6452 | FN=17 | FP=5
TF-IDF LogReg  : Recall=0.9459 | Precision=0.6863 | F1=0.7955 | FN=2  | FP=16

=== Performance by Language Style ===
direct                 (n=28): Macro-F1=0.6883 | Urgency Recall=1.0000 | Urgency FN=0
euphemistic            (n=11): Macro-F1=0.4237 | Urgency Recall=1.0000 | Urgency FN=0
distressed_fragmented  (n=10): Macro-F1=0.5012 | Urgency Recall=0.7778 | Urgency FN=2
third_person           (n=11): Macro-F1=0.7223 | Urgency Recall=1.0000 | Urgency FN=0

=== Performance by Message Length ===
short  (<12 words, n=9)  : Macro-F1=0.5778 | Urgency Recall=0.7500 | Urgency FN=2
medium (12-28 w,  n=31)  : Macro-F1=0.6451 | Urgency Recall=1.0000 | Urgency FN=0
long   (>28 w,    n=20)  : Macro-F1=0.6464 | Urgency Recall=1.0000 | Urgency FN=0


## Key Finding & Misclassified Examples

1. **Missed Urgent Cases (`Urgency FN = 2`) occurred exclusively in short, distressed-fragmented messages** (`syn-080`: *"need safe place now"* with p=0.412, and `syn-133`: *"need someone to talk to feel hopeless"* with p=0.290) where few explicit emergency unigrams/bigrams appeared.
2. **Support-category recall dropped sharply on euphemistic messages** (`Macro-F1 = 0.4237`), where survivors avoid explicit clinical or legal words (e.g., `syn-033`: *"He took what he wanted from me while I was half asleep..."* missed `sexual_violence_support`).
3. **Design implication for OneDoor:** Never deny or hide support options based on model confidence, always display uncertainty openly, and treat human review as essential.